# Phutball Transformer - Tabula Rasa 21x15

Trains from random init at full board size (21x15) only.
No ladder escalation, no escalation across board sizes; single fixed
board throughout. Uses the post-reachability network (10 input
channels, single-jump reachability mask exposed as a planning prior),
so this notebook is not checkpoint-compatible with pre-reachability
runs.

Runtime to A100 GPU or TPU v6e to Run all


In [ ]:
# Detect environment
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ or 'google.colab' in str(globals())
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# Install dependencies
if IN_COLAB:
    try:
        import jax
        if 'TPU' in str(jax.devices()):
            !pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
        else:
            raise Exception("No TPU")
    except:
        !pip install -q jax[cuda12_pip] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    !pip install -q flax optax wandb mctx

In [ ]:
# Verify devices
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Device count: {jax.device_count()}")

DEVICE_COUNT = jax.device_count()
DEVICE_TYPE = str(jax.devices()[0]).split(':')[0] if jax.devices() else 'cpu'
print(f"\nUsing {DEVICE_COUNT}x {DEVICE_TYPE}")

In [ ]:
# Mount Google Drive for checkpoints
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Full board ladder (same sizes as ladder run)
ALL_LADDER_SIZES = ((13, 7), (15, 9), (17, 11), (19, 13), (21, 15))

# Architecture params (must match ladder notebook)
D_MODEL = 512
N_LAYERS = 12

if IN_COLAB:
    CHECKPOINT_ROOT = "/content/drive/MyDrive/phutball_checkpoints"
else:
    CHECKPOINT_ROOT = "./checkpoints"

print(f"Board ladder: {ALL_LADDER_SIZES}")
print(f"Checkpoint root: {CHECKPOINT_ROOT}")

In [ ]:
# Wandb login (optional)
USE_WANDB = True

if USE_WANDB:
    import wandb
    wandb.login(relogin=True)

# === FRESH START vs RESUME ===
# Set FRESH_START = True when restarting training from scratch (new wandb run).
# Set FRESH_START = False (default) to resume a crashed/stopped run.
FRESH_START = True  # <-- Set True to start clean after code changes

In [ ]:
# Clone/update repo to value-head-ablation branch
REPO_URL = "https://github.com/echoname6/phutball-jax.git"
REPO_BRANCH = "value-head-ablation"
REPO_DIR = "/content/phutball-jax" if IN_COLAB else "./phutball-jax"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git fetch origin && git checkout {REPO_BRANCH} && git pull origin {REPO_BRANCH}
else:
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -3
!git rev-parse --abbrev-ref HEAD


In [ ]:
# === TRAINING CONFIG ===
# Hyperparams match the ladder notebook exactly for controlled comparison.
# The only difference: weights are reinitialized at each board size.

POS_ENCODING = "goal_distance"

# Self-play
BATCH_SIZE_GAMES = 64
NUM_SIMULATIONS = 32
GAMES_PER_ITER = 256

# Training
BATCH_SIZE_TRAIN = 256
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TRAIN_STEPS_PER_ITER = 1000
BUFFER_SIZE = 1_000_000
MIN_BUFFER_SIZE = 10_000

# Cosine LR schedule
COSINE_LR_ENABLED = True
LR_END = 1e-5
LR_WARMUP_ITERS = 10

DRAW_VALUE = 0.0
TEMP_THRESHOLD = 60
TEMP_FINAL = 0.1
CURRICULUM_ENABLED = False

# ELO evaluation (logging only — no longer drives escalation)
ELO_EVAL_ENABLED = True
ELO_EVAL_GAMES_PER_PERSPECTIVE = 10
ELO_EVAL_MAX_OPPONENTS = 5
ELO_EVAL_NUM_SIMULATIONS = 32

# ELO-based sim escalation — DISABLED (replaced by plateau escalation)
ELO_ESCALATION_ENABLED = False

# Policy-loss plateau escalation (same tiers as ladder)
PLATEAU_ESCALATION_ENABLED = True
PLATEAU_WINDOW = 20
PLATEAU_COMPARE_WINDOW = 20
PLATEAU_MIN_IMPROVEMENT = 0.005
PLATEAU_PATIENCE = 2
PLATEAU_MIN_ITERS_PER_RUNG = 50
PLATEAU_SIM_TIERS = (32, 48, 64)

# KL early-stopping: revert params on catastrophic policy collapse
KL_EARLY_STOP_ENABLED = True
KL_EARLY_STOP_THRESHOLD = 0.5
KL_EARLY_STOP_MAX_REVERTS = 3

# Per-board-size iteration budget
ITERS_PER_BOARD = 10_000
CHECKPOINT_EVERY = 10

# Notifications
NTFY_TOPIC = "phutball-blank-21x15"
HEARTBEAT_MINUTES = 30

WANDB_PROJECT = "phutball-blank-21x15"

print(f"Board: 21x15 (fixed)")
print(f"Iters: {ITERS_PER_BOARD}")
print(f"Plateau escalation: {PLATEAU_ESCALATION_ENABLED} (tiers={PLATEAU_SIM_TIERS}, patience={PLATEAU_PATIENCE})")
print(f"KL early-stop: {KL_EARLY_STOP_ENABLED} (threshold={KL_EARLY_STOP_THRESHOLD})")

In [ ]:
# Imports
import numpy as np
import time

from phutball_env_jax import (
    PhutballState, EnvConfig, reset, step, get_legal_actions,
    state_to_network_input, render_board
)
from network import PhutballTransformer, create_transformer_network
from train_batched import TransformerTrainer, TrainConfig

print("Imports OK")

## Training Loop

Single-board tabula rasa run. No board-size escalation; weights stay
fresh init at 21x15 through the full ITERS_PER_BOARD budget. Plateau
sim escalation still drives intra-rung simulation count.


In [ ]:
# Fixed at 21x15 — no ladder escalation
BOARD_INDEX = 4  # 0=13x7, 1=15x9, 2=17x11, 3=19x13, 4=21x15

ROWS, COLS = ALL_LADDER_SIZES[BOARD_INDEX]
print(f"Tabula rasa at {ROWS}x{COLS}")

In [ ]:
# Setup checkpoint dir and wandb for this board size
CHECKPOINT_BASE = os.path.join(
    CHECKPOINT_ROOT,
    f"blank_{ROWS}x{COLS}_d{D_MODEL}_l{N_LAYERS}"
)
os.makedirs(CHECKPOINT_BASE, exist_ok=True)

# Handle fresh start vs resume
WANDB_RUN_ID = None
WANDB_ID_FILE = os.path.join(CHECKPOINT_BASE, "wandb_run_id.txt")

if FRESH_START:
    if os.path.exists(WANDB_ID_FILE):
        old_id = open(WANDB_ID_FILE).read().strip()
        os.remove(WANDB_ID_FILE)
        print(f"Cleared old wandb run ID: {old_id}")
    print("FRESH START: new wandb run will be created")
else:
    if os.path.exists(WANDB_ID_FILE):
        with open(WANDB_ID_FILE) as f:
            WANDB_RUN_ID = f.read().strip()
        print(f"Resuming wandb run: {WANDB_RUN_ID}")
    else:
        print("Fresh wandb run")

print(f"Checkpoints: {CHECKPOINT_BASE}")

In [ ]:
# Create config — single board size, no ladder escalation
config = TrainConfig(
    rows=ROWS,
    cols=COLS,

    num_channels=D_MODEL,
    num_res_blocks=N_LAYERS,
    pos_encoding=POS_ENCODING,

    # Self-play
    batch_size_games=BATCH_SIZE_GAMES,
    num_simulations=NUM_SIMULATIONS,
    games_per_iteration=GAMES_PER_ITER,
    temp_threshold=TEMP_THRESHOLD,
    temp_final=TEMP_FINAL,

    # Training
    batch_size_train=BATCH_SIZE_TRAIN,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    train_steps_per_iteration=TRAIN_STEPS_PER_ITER,
    cosine_lr_enabled=COSINE_LR_ENABLED,
    lr_end=LR_END,
    lr_warmup_iters=LR_WARMUP_ITERS,
    draw_value=DRAW_VALUE,
    buffer_size=BUFFER_SIZE,
    min_buffer_size=MIN_BUFFER_SIZE,

    curriculum_enabled=CURRICULUM_ENABLED,

    # ELO evaluation (logging only)
    elo_eval_enabled=ELO_EVAL_ENABLED,
    elo_eval_games_per_perspective=ELO_EVAL_GAMES_PER_PERSPECTIVE,
    elo_eval_max_opponents=ELO_EVAL_MAX_OPPONENTS,
    elo_eval_num_simulations=ELO_EVAL_NUM_SIMULATIONS,

    # ELO-based sim escalation — disabled
    elo_escalation_enabled=ELO_ESCALATION_ENABLED,

    # Policy-loss plateau escalation
    plateau_escalation_enabled=PLATEAU_ESCALATION_ENABLED,
    plateau_window=PLATEAU_WINDOW,
    plateau_compare_window=PLATEAU_COMPARE_WINDOW,
    plateau_min_improvement=PLATEAU_MIN_IMPROVEMENT,
    plateau_patience=PLATEAU_PATIENCE,
    plateau_min_iters_per_rung=PLATEAU_MIN_ITERS_PER_RUNG,
    plateau_sim_tiers=PLATEAU_SIM_TIERS,

    # KL early-stopping
    kl_early_stop_enabled=KL_EARLY_STOP_ENABLED,
    kl_early_stop_threshold=KL_EARLY_STOP_THRESHOLD,
    kl_early_stop_max_reverts=KL_EARLY_STOP_MAX_REVERTS,

    # No board ladder — fixed at 21x15
    board_ladder_enabled=False,

    num_iterations=ITERS_PER_BOARD,

    checkpoint_dir=CHECKPOINT_BASE,
    checkpoint_every=CHECKPOINT_EVERY,
    fresh_start=FRESH_START,

    ntfy_topic=NTFY_TOPIC,
    heartbeat_minutes=HEARTBEAT_MINUTES,

    use_wandb=USE_WANDB,
    wandb_project=WANDB_PROJECT,
    wandb_run_name=f"blank_{ROWS}x{COLS}_d{D_MODEL}_l{N_LAYERS}",
    wandb_run_id=WANDB_RUN_ID,
)

print(f"Config created for {ROWS}x{COLS} tabula rasa (no ladder)")
print(f"Plateau sim escalation: tiers={config.plateau_sim_tiers}, patience={config.plateau_patience}")
print(f"KL early-stop: threshold={config.kl_early_stop_threshold}, max_reverts={config.kl_early_stop_max_reverts}")
print(f"fresh_start = {config.fresh_start}")

In [ ]:
# Create trainer (random init — no checkpoint loading)
trainer = TransformerTrainer(config)

# Resume from checkpoint
if not FRESH_START:
    import glob
    import re

    best_ckpt = None
    best_iter = -1

    for ckpt in glob.glob(os.path.join(CHECKPOINT_BASE, "checkpoint_*.pkl")):
        match = re.search(r'checkpoint_(\d+)', os.path.basename(ckpt))
        if match:
            it = int(match.group(1))
            if it >= best_iter:
                best_iter = it
                best_ckpt = ckpt

    if best_ckpt:
        print(f"Resuming from: {best_ckpt} (iteration {best_iter})")
        trainer.load_checkpoint(best_ckpt)
        trainer.config.fresh_start = True
        print(f"Resumed at iteration {trainer.iteration}")
    else:
        print(f"No checkpoints found — starting fresh at {ROWS}x{COLS}")
else:
    print(f"FRESH START: training from scratch at {ROWS}x{COLS}")

param_count = sum(x.size for x in jax.tree_util.tree_leaves(trainer.params))
print(f"Transformer parameters: {param_count:,}")

In [ ]:
# Train!
trainer.train()

## Evaluation

In [ ]:
# Evaluate vs random
if trainer:
    print(f"\nEvaluation for {ROWS}x{COLS} tabula rasa:")
    win_rate, stats = trainer.evaluate_vs_random_batched()
    print(f"Win rate: {win_rate:.1%}")

In [ ]:
# List checkpoints
import glob

print(f"Checkpoints for {ROWS}x{COLS} tabula rasa:")
ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_BASE, "*.pkl")))
print(f"Total: {len(ckpts)}")
for c in ckpts[-5:]:
    print(f"  - {os.path.basename(c)}")